# OCR Processor

Mistral Document AI API comes with a Document OCR (Optical Character Recognition) processor, powered by our latest OCR model `mistral-ocr-latest`, which enables you to extract text and structured content from PDF documents.

This notebook demonstrates how to upload a PDF file to Mistral Cloud and get the OCR results from the uploaded PDF by retrieving a signed url.


In [ ]:
import getpass

MISTRAL_API_KEY = getpass.getpass("Enter your API KEY: ")

In [ ]:
from mistralai import Mistral

client = Mistral(api_key=MISTRAL_API_KEY)

In [ ]:
# Upload a file to Mistral Cloud

from pathlib import Path

path = Path(input())

uploaded_pdf = client.files.upload(
    file={
        "file_name": path.name,
        "content": open(path, "rb"),
    },
    purpose="ocr"
)  

In [ ]:
# Get a signed url to access the file

signed_url = client.files.get_signed_url(file_id=uploaded_pdf.id)

In [ ]:
# Query the OCR endpoint with the signed url

ocr_response = client.ocr.process(
    model="mistral-ocr-latest",
    document={
        "type": "document_url",
        "document_url": signed_url.url,
    },
    include_image_base64=False
)

# Check all available params here: https://docs.mistral.ai/api/endpoint/ocr

In [ ]:
# Take a look at the pages

for i, page in enumerate(ocr_response.pages):
    print(f"[Page {i}]")
    print(page.markdown[:200]+"...\n\n")

In [ ]:
# Export to markdown
md_path = path.with_suffix(".md")
with open(md_path, "w", encoding="utf-8") as f:
    for page in ocr_response:
        f.write(page.markdown)
        f.write("\n\n")  # page separator

In [ ]:
# Delete the pdf file from Mistral cloud unless you wish to reuse it later:

client.files.delete(file_id=uploaded_pdf.id)

---